## Student Performance Indicator

#### Life cycle of Machine learning Project

- Understanding the Problem Statement
- Data Collection
- Data Checks to perform
- Exploratory data analysis
- Data Pre-Processing
- Model Training
- Choose best model

### 1) Problem statement
- This project understands how the student's performance (test scores) is affected by other variables such as Gender, Ethnicity, Parental level of education, Lunch and Test preparation course.


### 2) Data Collection
- Dataset Source - https://www.kaggle.com/datasets/spscientist/students-performance-in-exams?datasetId=74977
- The data consists of 8 column and 1000 rows.

In [5]:
%load_ext autoreload
%autoreload 2

In [3]:
%pwd

'f:\\Desktop\\DATA SCIENCE\\Topics (Left)\\MLOPs\\CI-CD + Monitoring + Orchestration\\Tutorial'

In [2]:
%cd ..

f:\Desktop\DATA SCIENCE\Topics (Left)\MLOPs\CI-CD + Monitoring + Orchestration\Tutorial


In [ ]:
# import warnings
import os
import sys
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from dataclasses import dataclass
from pathlib import Path
from mlops.constants import CONFIG_FILE_PATH
from mlops.utils.common import read_yaml, create_dir, save_json, save_object, get_env
from mlops.exception import CustomException
import mlflow


True

In [5]:
from mlops.logger import setup_logger
logging = setup_logger()

In [7]:
# entity

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    train_data_file: Path
    tranied_model_path: Path
    model_report: Path
    

In [8]:
# configuration

class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH):
        self.config = read_yaml(config_file_path)
        create_dir([self.config.artifacts_root])
    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        train_data_file = self.config.data_transformation.transformed_train_data
        create_dir([training.root_dir])

        training_config = TrainingConfig(
            root_dir = training.root_dir,
            train_data_file = Path(train_data_file),
            tranied_model_path = Path(training.trained_model_path),
            model_report=Path(training.model_report)
        )

        return training_config
        


In [ ]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.lr = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.y_train_pred = None
        self.y_test_pred = None
        self._is_trained = False

    def train(self):
        """This function is used for model training"""

        try:

            logging.info("Training started --->")
            logging.info("Loading of transformed train data started--->")
            
            train_arr = np.load(self.config.train_data_file)
            
            logging.info("Transformed train data loaded successfully.")
            
            X = train_arr[:,:-1]
            y = train_arr[:,-1]
            
            logging.info("Starting train test split---->")
            
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                X,y,test_size=0.2,random_state=42)

            logging.info("Train-test-split is done successfully.")

            self.lr = LinearRegression()

            logging.info("Model fitting started -->")

            self.lr.fit(self.X_train, self.y_train) # Train/fitting the model
            self._is_trained = True

            logging.info("Model fitting completed.")
            logging.info("Predictions are made.")

        except Exception as e:
            raise CustomException(e,sys)

    
    @staticmethod
    def _evaluate_model(true, predicted):
        model_report = {}
        mae = mean_absolute_error(true, predicted)
        mse = mean_squared_error(true, predicted)
        rmse = np.sqrt(mean_squared_error(true, predicted))
        r2_value = r2_score(true, predicted)
        
        model_report = {
            "mae": mae,
            "rmse": rmse,
            "r2_score": r2_value
        }
        return model_report
    
    # def evaluate(self):
    #     """This function is for evaluation"""

    #     try:
    
    #         if not self._is_trained:
    #             raise RuntimeError("You must call train() before evaluate().")

    #         # Make predictions
    #         self.y_train_pred = self.lr.predict(self.X_train)
    #         self.y_test_pred = self.lr.predict(self.X_test)
            

    #         # Evaluate Train and Test dataset
    #         y_train_model_report = self._evaluate_model(self.y_train, self.y_train_pred)
    #         y_test_model_report = self._evaluate_model(self.y_test, self.y_test_pred)

    #         save_object(file_path=self.config.tranied_model_path,
    #                     obj=self.lr)
            
    #         logging.info(f"Trained model is saved at {self.config.tranied_model_path}")
            
    #         report = {"train": y_train_model_report,"test": y_test_model_report}

    #         save_json(data=report, file_path=self.config.model_report)

    #         logging.info(f"Model report of train and test set is saved at {self.config.model_report}")

    #     except Exception as e:
    #         raise CustomException(e,sys)
        
    def evaluate(self):
        """This function is for evaluation"""

        if not self._is_trained:
                raise RuntimeError("You must call train() before evaluate().")


        try:

            # Make predictions
            self.y_train_pred = self.lr.predict(self.X_train)
            self.y_test_pred = self.lr.predict(self.X_test)
            

            # Evaluate Train and Test dataset
            y_train_model_report = self._evaluate_model(self.y_train, self.y_train_pred)
            y_test_model_report = self._evaluate_model(self.y_test, self.y_test_pred)

            save_object(file_path=self.config.tranied_model_path,
                        obj=self.lr)
            
            logging.info(f"Trained model is saved at {self.config.tranied_model_path}")
            
            report = {"train": y_train_model_report,"test": y_test_model_report}
            self.report = report

            save_json(data=report, file_path=self.config.model_report)

            logging.info(f"Model report of train and test set is saved at {self.config.model_report}")
            return report

        except Exception as e:
            raise CustomException(e,sys)

    def log_to_mlflow(
        self,
        registered_model_name=None,
        experiment_name=None,
        run_name=None,
        # tracking_uri=os.environ["MLFLOW_TRACKING_URI"]
        tracking_uri = None
    ):
        """Log model metrics, report artifact, and model to MLflow."""

        if not self._is_trained:
            raise RuntimeError("You must call train() before log_to_mlflow().")

        if self.report is None:
            self.evaluate()

        try:
            if tracking_uri:
                mlflow.set_tracking_uri(tracking_uri)

            if experiment_name:
                mlflow.set_experiment(experiment_name)

            metrics = {
                "train_mae": self.report["train"]["mae"],
                "train_rmse": self.report["train"]["rmse"],
                "train_r2_score": self.report["train"]["r2_score"],
                "test_mae": self.report["test"]["mae"],
                "test_rmse": self.report["test"]["rmse"],
                "test_r2_score": self.report["test"]["r2_score"],
            }

            with mlflow.start_run(run_name=run_name):
                mlflow.log_metrics(metrics)
                mlflow.log_artifact(str(self.config.model_report))

                model_info = mlflow.sklearn.log_model(
                    sk_model=self.lr,
                    artifact_path="model",
                    registered_model_name=registered_model_name,
                )

            logging.info("Model metrics, report, and model logged to MLflow successfully.")
            return model_info

        except Exception as e:
            raise CustomException(e, sys)



        

In [ ]:
# pipeline

try:
    logging.info("Model training pipeline started ---->")
    
    config = ConfigurationManager()
    model_training_config = config.get_training_config()
    model_trainer = Training(config=model_training_config)
    model_trainer.train()
    model_trainer.evaluate()
    model_trainer.log_to_mlflow(
    registered_model_name="student-performance-model",
    experiment_name="student-performance-training",
    run_name="linear-regression",
    tracking_uri=get_env("MLFLOW_TRACKING_URI")
    )
    
    logging.info("Model training pipeline executed successfully")

except Exception as e:
    raise CustomException(e,sys)

[ 2026-07-06 12:00:00,452 ] 4 3515248405.py root - INFO - Model training pipeline started ---->
[ 2026-07-06 12:00:00,457 ] 34 common.py root - INFO - yaml file: config\config.yaml loaded sucessfully.
[ 2026-07-06 12:00:00,457 ] 48 common.py root - INFO - created directory at artifacts
[ 2026-07-06 12:00:00,465 ] 48 common.py root - INFO - created directory at artifacts/training
[ 2026-07-06 12:00:00,465 ] 18 2571346593.py root - INFO - Training started --->
[ 2026-07-06 12:00:00,465 ] 19 2571346593.py root - INFO - Loading of transformed train data started--->
[ 2026-07-06 12:00:00,473 ] 23 2571346593.py root - INFO - Transformed train data loaded successfully.
[ 2026-07-06 12:00:00,473 ] 28 2571346593.py root - INFO - Starting train test split---->
[ 2026-07-06 12:00:00,516 ] 33 2571346593.py root - INFO - Train-test-split is done successfully.
[ 2026-07-06 12:00:00,518 ] 37 2571346593.py root - INFO - Model fitting started -->
[ 2026-07-06 12:00:00,533 ] 42 2571346593.py root - INFO

🏃 View run linear-regression at: https://dagshub.com/Biki-98/mlops.mlflow/#/experiments/0/runs/3c1176849dad46d1afb4e21fcf88e4fa
🧪 View experiment at: https://dagshub.com/Biki-98/mlops.mlflow/#/experiments/0


[ 2026-07-06 12:00:55,518 ] 170 2571346593.py root - INFO - Model metrics, report, and model logged to MLflow successfully.
[ 2026-07-06 12:00:55,522 ] 17 3515248405.py root - INFO - Model training pipeline executed successfully
